In [7]:
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime, timezone
import json
import sqlite3
import hashlib


@dataclass(eq=True, frozen=True)
class VesselMetadataKey:
    unique_vessel_id: str
    obs_time: datetime

def sort_dict_list_by_keys(input: list, output: list) -> list:
    for e in input:
        if isinstance(e, dict):
            d = {}
            output.append(d)
            sort_dict_by_keys(e, d)
        elif isinstance(e, list):
            l = []
            output.append(l)
            sort_dict_list_by_keys(e, l)
        else:
            output.append(e)

def sort_dict_by_keys(input: dict, output: dict, *,
                      exclude_keys: tuple = ('time', 'fileType', 'submissionInfo', 'dataProcessed')) -> dict:
    for k, v in sorted(input.items()):
        if k in exclude_keys:
            continue
        if isinstance(v, dict):
            output[k] = {}
            sort_dict_by_keys(v, output[k])
        elif isinstance(v, list):
            output[k] = []
            sort_dict_list_by_keys(v, output[k])
        else:
            output[k] = v
    return output

def get_unique_vessel_id(data: dict) -> str|None:
    if 'platform' in data:
        platform = data['platform']
        if 'uniqueID' in platform:
            uniqueId = platform['uniqueID']
            if not isinstance(uniqueId, str):
                raise ValueError(f"Expected uniqueID to be of type str but is of type {type(uniqueId)}")
            return uniqueId
    if 'trustedNode' in data:
        trustedNode = data['trustedNode']
        if 'uniqueVesselID' in trustedNode:
            uniqueVesselId = trustedNode['uniqueVesselID']
            if not isinstance(uniqueVesselId, str):
                raise ValueError(f"Expected uniqueVesselId to be of type str but is of type {type(uniqueVesselId)}")
            return uniqueVesselId
    return None

def get_start_end_times(data: dict) -> tuple[datetime, datetime]|None:
    if 'time' in data:

    return None

In [43]:
# doc_root should be local path to https://github.com/CCOMJHC/csbschema/tree/main/docs/IHO
# doc_root: Path = Path('../docs/IHO')
# example_docs: list[str] = [
#     'b12_v3_1_0_example-2023-08.json'
# ]


In [2]:
doc_root: Path = Path('example1')
for doc in doc_root.glob('*.json'):
    print(str(doc))

example1/A15260F9-CE28-41FB-AED8-B91AB52D1B78.json
example1/F655ADEC-E1D8-4F79-BF98-0C05D58FFF52.json
example1/0A99C49B-8262-40FC-BFB6-2F21848B0299.json
example1/698F8CE4-C8D5-4FD2-8AEC-F64767258CB9.json
example1/5BAB57D1-7D21-4E77-8497-2524E718D012.json


In [10]:
vessel_meta: dict[VesselMetadataKey, dict] = {}

for doc in doc_root.glob('*.json'):
    # print(str(doc))
    with doc.open(mode='rt') as f:
        doc_data: dict = json.load(f)

    try:
        uniqueId = get_unique_vessel_id(doc_data)
    except ValueError as e:
        print(f"Unable to read unique ID for file {str(doc)} due to error {str(e)}, skipping...")
        continue
    if uniqueId is None:
        print(f"No unique ID for file {str(doc)}, skipping...")
        continue

    print(f"Unique Id for file {str(doc)} is {uniqueId}")

    # print(f"raw doc_meta: {json.dumps(doc_meta)}\n\n")
    doc_meta: dict = sort_dict_by_keys(doc_data, {})
    # print(f"sorted doc_meta: {json.dumps(doc_meta)}\n\n")
    key = VesselMetadataKey(
        unique_vessel_id=uniqueId,
        obs_time=datetime.now(timezone.utc)
    )
    vessel_meta[key] = doc_meta

print(vessel_meta)

Unique Id for file example1/A15260F9-CE28-41FB-AED8-B91AB52D1B78.json is OFM-72b748d0-9890-11f0-bdd5-b1670927a4f1
Unique Id for file example1/F655ADEC-E1D8-4F79-BF98-0C05D58FFF52.json is SIGNALK-ac020bdf-5c0e-4c82-844f-1db2bc73383a
Unique Id for file example1/0A99C49B-8262-40FC-BFB6-2F21848B0299.json is PGS-834f85c2-0999-11eb-a100-98be942a5b5a
Unique Id for file example1/698F8CE4-C8D5-4FD2-8AEC-F64767258CB9.json is AQM-687ce9f49cea48-68471861
Unique Id for file example1/5BAB57D1-7D21-4E77-8497-2524E718D012.json is AQM-687ce9f49cea48-68471861
{VesselMetadataKey(unique_vessel_id='OFM-72b748d0-9890-11f0-bdd5-b1670927a4f1', obs_time=datetime.datetime(2026, 2, 11, 23, 9, 55, 329332, tzinfo=datetime.timezone.utc)): {'convention': 'XYZ CSB 3.0', 'correctors': {'draftApplied': 'False', 'motionOffsetsApplied': 'False', 'positionOffsetDocumented': 'False', 'positionReferencePoint': 'GNSS', 'soundSpeedDocumented': 'False'}, 'crs': {'horizontal': {'type': 'EPSG', 'value': 4326}, 'vertical': 'Trans

In [48]:
con = sqlite3.connect('vessel_meta.db')
cur = con.cursor()
cur.execute('CREATE TABLE vessels(unique_vessel_id TEXT, obs_time DATETIME, hash TEXT, metadata JSON, PRIMARY KEY(unique_vessel_id, obs_time))')

In [49]:
for k, v in vessel_meta.items():
    m = hashlib.sha3_256()
    metadata: str = json.dumps(v)
    m.update(bytes(metadata, 'utf-8'))
    metadata_hash: str = m.hexdigest()
    data = [k.unique_vessel_id, k.obs_time, metadata_hash, metadata]
    cur.execute("INSERT INTO vessels VALUES(?, ?, ?, ?)", data)

In [50]:
res = cur.execute('SELECT * FROM vessels')
res.fetchall()

[('SEAID-e8c469f8-df38-11e5-b86d-9a79f06e9478',
  '2026-01-16 01:45:09.908715+00:00',
  '548eedb81237e39837856a5b4955e828a6ba4b1d029833bae930218695425330',
  '{"platform": {"IDNumber": "369958000", "IDType": "MMSI", "contributorComments": "On 2022-03-08, at 20:30 UTC, the echo sounder lost bottom tracking after the vessel crossed another vessel\'s wake.", "dataProcessed": true, "length": 65, "name": "White Rose of Drachs", "positionOffsetsDocumented": true, "sensors": [{"draft": 1.4, "draftUncert": 0.2, "frequency": 200000, "make": "Garmin", "model": "GT-50", "position": [4.2, 0.0, 5.4], "type": "Sounder"}, {"make": "Litton Marine Systems", "model": "LMX420", "type": "GNSS"}], "soundSpeedDocumented": true, "type": "Private vessel", "uniqueID": "SEAID-e8c469f8-df38-11e5-b86d-9a79f06e9478"}, "processing": [{"destination": "EPSG:8252", "method": "GeoTrans", "original": "EPSG:4326", "timestamp": "2023-02-14T02:00:00.0000Z", "type": "CRSChange"}, {"datum": "CANNORTH2016v1HyVSEP_NAD83v6_CD",